# Student 360 Workshop - Build and Harden the AI Agents
## Notebook 04 (Phase 4 - Build the AI experience)

_Icons: 🛠️ Action  📌 Note  🔹 Info  🎓 Higher-ed context  ✅ Best practice_

---
### What this notebook does
1. 🛠️ Creates **two agents from one specification pattern** - an *Advisor Assistant* (College of Engineering only) and a *Student Success Analyst* (institution-wide) - each with a structured tool, an unstructured tool and a chart tool
2. 🛠️ Grants the **layer-2 agent permission** that completes the four-layer access model
3. 🛠️ Tests each agent **as each persona** and shows the data layer enforcing scope
4. 🛠️ Adds **cost control**: attribution tags, a warehouse statement timeout, a spending budget and a per-user quota (monitoring mode)

### Why it matters
An agent is a *spec*, not a prompt: tools, instructions and guardrails declared in one versioned object. Keeping it declarative means you can review it, version it and promote it - the same discipline as the data layer.

**Estimated time:** 40 min  |  **Prerequisite:** Notebook 03

In [ ]:
%%sql -r set_context
USE ROLE STUDENT360_LAB_ADMIN;
USE DATABASE STUDENT360_LAB;
USE SCHEMA AGENTS;
USE WAREHOUSE STUDENT360_LAB_WH;
SELECT CURRENT_ROLE(), CURRENT_SCHEMA();

## Step 1: Anatomy of an Agent Specification

🛠️ The spec below has five parts. Read them before you run the cell:

| Section | Purpose | Best practice |
|---|---|---|
| `orchestration.budget` | Caps seconds and tokens per question | Always set one - a runaway agent is a cost incident |
| `instructions.response` | *How* to answer: tone, format, citations, guardrails | Put non-negotiables here: "never disclose contact details", "a risk score is a triage signal" |
| `instructions.orchestration` | *Which tool* to use for which question | Route explicitly; vague routing is the #1 cause of wrong tool choice |
| `tools` | What the agent can call | `description` is how the model picks a tool - write it like a job description |
| `tool_resources` | Where each tool points | Point structured tools at the **semantic view**, unstructured tools at a **search service** |

✅ **Best practice - two agents, not one super-agent.** The Advisor Assistant and the Student Success Analyst share the same semantic view and tools, but the **data they can reach is different** because their roles differ (Notebook 03). Scope is enforced by the data layer, so the two agents can use the *same code*.

### Step 2: Create the Advisor Assistant (College of Engineering)

In [ ]:
%%sql -r create_advisor_agent
CREATE OR REPLACE AGENT STUDENT360_LAB.AGENTS.ADVISOR_ASSISTANT
  COMMENT = 'Advisor Assistant for the College of Engineering - caseload prioritisation, scoped to Engineering rows and notes'
  PROFILE = '{"display_name": "Advisor Assistant (Engineering)", "color": "green"}'
  FROM SPECIFICATION
$$
models:
  orchestration: auto

orchestration:
  budget:
    seconds: 45
    tokens: 16000

instructions:
  response: |
    You are the Advisor Assistant for Lakeside University's College of Engineering.
    You help academic advisors prioritise outreach to their students.
    - Be concise. Lead with the student's name, STUDENT_ID and the risk tier.
    - Always explain WHY a student is at risk using TOP_RISK_DRIVERS. Never give a risk score without its drivers.
    - The risk score is a triage signal for outreach. Never describe a student as certain to leave.
    - Never disclose or infer contact details, dates of birth, or protected characteristics.
    - If the question spans structured data and advising notes, use BOTH tools and cite which source each fact came from.
  orchestration: |
    - Numbers, rankings, lists of students, GPA, holds, LMS and campus data: use student_analytics.
    - Why a student is struggling, advisor-noted concerns, themes in conversations: use advising_notes.
    - Advisors name themselves in the question (for example "my caseload" or "Dana Whitfield's students"). Filter on ASSIGNED_ADVISOR. If no advisor name is given, ask which advisor the caseload belongs to.
    - Questions about students outside the Engineering college will not return data. Say so plainly.
  sample_questions:
    - question: "Who are the 10 students on Dana Whitfield's caseload I should reach out to first, and why?"
    - question: "Which of my students have a registration hold?"
    - question: "What are students in my college saying in advising notes about finances?"

tools:
  - tool_spec:
      type: "cortex_analyst_text_to_sql"
      name: "student_analytics"
      description: "Query structured student data: students, enrollments, GPA, risk scores and drivers, financial need, holds, LMS engagement and campus activity. Use for any question about numbers, counts, rankings or lists of students."
  - tool_spec:
      type: "cortex_search"
      name: "advising_notes"
      description: "Search PII-redacted advising notes for qualitative context such as what students and advisors are reporting. Use for why questions and for themes."
  - tool_spec:
      type: "data_to_chart"
      name: "data_to_chart"
      description: "Create a chart from query results when the user asks for a visual."

tool_resources:
  student_analytics:
    semantic_view: "STUDENT360_LAB.SEMANTIC.STUDENT_360_SV"
    execution_environment:
      type: warehouse
      warehouse: "STUDENT360_LAB_WH"
  advising_notes:
    name: "STUDENT360_LAB.AGENTS.ADVISING_NOTES_SEARCH_ENGINEERING"
    max_results: "5"
$$;

### Step 3: Create the Student Success Analyst (institution-wide)

In [ ]:
%%sql -r create_ir_agent
CREATE OR REPLACE AGENT STUDENT360_LAB.AGENTS.STUDENT_SUCCESS_ANALYST
  COMMENT = 'Institutional Research assistant - institution-wide aggregate analysis, PII masked'
  PROFILE = '{"display_name": "Student Success Analyst", "color": "blue"}'
  FROM SPECIFICATION
$$
models:
  orchestration: auto

orchestration:
  budget:
    seconds: 45
    tokens: 16000

instructions:
  response: |
    You are the Student Success Analyst for Lakeside University Institutional Research.
    You answer institution-wide questions about retention, academics, engagement, finances and advising.
    - Be concise and data-driven. Lead with the number, then the breakdown.
    - Show percentages with one decimal place. Show money in whole dollars.
    - Always state the population and the term you are reporting on (Fall 2026, week 10).
    - Aggregate analysis only: do not name individual students unless asked for a specific student's risk drivers.
    - Never disclose or infer contact details or dates of birth.
    - If the question spans structured data and advising notes, use BOTH tools and cite which source each fact came from.
  orchestration: |
    - Counts, rates, averages, rankings, comparisons across colleges, cohorts or course sections: use student_analytics.
    - Qualitative themes, what advisors are hearing, why students are struggling: use advising_notes.
    - For questions about risk drivers across the population, use student_analytics on TOP_RISK_DRIVERS.
  sample_questions:
    - question: "How many high-risk students are in each college, and what is the at-risk rate?"
    - question: "Which courses have the highest DFW rate this term?"
    - question: "What are advisors hearing from students who are considering leaving?"

tools:
  - tool_spec:
      type: "cortex_analyst_text_to_sql"
      name: "student_analytics"
      description: "Query structured student data: students, enrollments, GPA, risk scores and drivers, financial need, holds, LMS engagement and campus activity. Use for any question about numbers, counts, rankings or lists of students."
  - tool_spec:
      type: "cortex_search"
      name: "advising_notes"
      description: "Search PII-redacted advising notes for qualitative context such as what students and advisors are reporting. Use for why questions and for themes."
  - tool_spec:
      type: "data_to_chart"
      name: "data_to_chart"
      description: "Create a chart from query results when the user asks for a visual."

tool_resources:
  student_analytics:
    semantic_view: "STUDENT360_LAB.SEMANTIC.STUDENT_360_SV"
    execution_environment:
      type: warehouse
      warehouse: "STUDENT360_LAB_WH"
  advising_notes:
    name: "STUDENT360_LAB.AGENTS.ADVISING_NOTES_SEARCH"
    max_results: "5"
$$;

In [ ]:
%%sql -r describe_agents
DESCRIBE AGENT STUDENT360_LAB.AGENTS.ADVISOR_ASSISTANT;
DESCRIBE AGENT STUDENT360_LAB.AGENTS.STUDENT_SUCCESS_ANALYST;

## Step 4: Layer 2 - Grant `USAGE ON AGENT`

🛠️ This completes the four-layer model from Notebook 03. Until this grant exists, the consumer role can see the semantic view but **cannot invoke the agent** - a common and confusing failure in real deployments.

🔹 Notice that the Advisor Assistant is granted only to the Engineering advisor role, and the Student Success Analyst only to the IR role. Nothing else changes.

In [ ]:
%%sql -r grant_agents
USE ROLE ACCOUNTADMIN;
GRANT USAGE ON AGENT STUDENT360_LAB.AGENTS.ADVISOR_ASSISTANT          TO ROLE STUDENT360_LAB_ADVISOR_ENGINEERING;
GRANT USAGE ON AGENT STUDENT360_LAB.AGENTS.STUDENT_SUCCESS_ANALYST    TO ROLE STUDENT360_LAB_IR_ANALYST;
USE ROLE STUDENT360_LAB_ADMIN;
SELECT 'USAGE ON AGENT granted (layer 2)' AS STATUS;

## Step 5: Test Each Agent as Each Persona

🛠️ Same question, different persona, different answer. Every call goes through `SNOWFLAKE.CORTEX.DATA_AGENT_RUN`, the same API that CoWork and custom apps use, so this is the real path.

> 🔹 `DATA_AGENT_RUN` returns the agent's full event stream (thinking, tool calls, tool results, final text). The final answer is the item with `type = 'text'`.

In [ ]:
%%sql -r test_advisor_agent
USE ROLE STUDENT360_LAB_ADVISOR_ENGINEERING;
USE SECONDARY ROLES NONE;
USE WAREHOUSE STUDENT360_LAB_WH;
SELECT TRY_PARSE_JSON(SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
  'STUDENT360_LAB.AGENTS.ADVISOR_ASSISTANT',
  $${"messages": [{"role": "user", "content": [{"type": "text", "text": "I am advisor Dana Whitfield. Who are the 5 students on my caseload I should reach out to first, and why?"}]}], "stream": false}$$
)) AS RESP;

In [ ]:
%%sql -r test_ir_agent
USE ROLE STUDENT360_LAB_IR_ANALYST;
USE SECONDARY ROLES NONE;
USE WAREHOUSE STUDENT360_LAB_WH;
SELECT TRY_PARSE_JSON(SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
  'STUDENT360_LAB.AGENTS.STUDENT_SUCCESS_ANALYST',
  $${"messages": [{"role": "user", "content": [{"type": "text", "text": "How many high-risk students are in each college, and what is the at-risk rate?"}]}], "stream": false}$$
)) AS RESP;

In [ ]:
%%sql -r test_guardrail
USE ROLE STUDENT360_LAB_IR_ANALYST;
USE SECONDARY ROLES NONE;
USE WAREHOUSE STUDENT360_LAB_WH;
SELECT TRY_PARSE_JSON(SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
  'STUDENT360_LAB.AGENTS.STUDENT_SUCCESS_ANALYST',
  $${"messages": [{"role": "user", "content": [{"type": "text", "text": "What is the email address and phone number of Maria Garcia?"}]}], "stream": false}$$
)) AS RESP;

In [ ]:
%%sql -r test_no_access
USE ROLE STUDENT360_LAB_ADVISOR_ENGINEERING;
USE SECONDARY ROLES NONE;
USE WAREHOUSE STUDENT360_LAB_WH;
-- Expect: error 399513 - agent does not exist or access is not authorized (layer 2 is missing for this role)
SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
  'STUDENT360_LAB.AGENTS.STUDENT_SUCCESS_ANALYST',
  $${"messages": [{"role": "user", "content": [{"type": "text", "text": "How many students are there?"}]}], "stream": false}$$
);

## Step 6: Cost Control - Observe, Attribute, Constrain

🛠️ The reference pattern for AI spend is **Observe → Attribute → Control**. Each step is a different Snowflake primitive:

| Step | Primitive | What it answers |
|---|---|---|
| **Observe** | `SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY` | Which agent, which user, how many tokens, how many credits |
| **Attribute** | A `TAG` on the agents and warehouse | Which department to charge |
| **Control** | Warehouse statement timeout, a **budget** with a spending limit and notification threshold, a **per-user quota** | Hard limits before a surprise invoice |

In [ ]:
%%sql -r observe_agents
SELECT START_TIME, USER_NAME, AGENT_NAME, TOKENS, TOKEN_CREDITS
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY
WHERE AGENT_NAME IN ('ADVISOR_ASSISTANT', 'STUDENT_SUCCESS_ANALYST')
ORDER BY START_TIME DESC
LIMIT 20;

In [ ]:
%%sql -r create_tag
CREATE TAG IF NOT EXISTS STUDENT360_LAB.AGENTS.COST_CENTER
  ALLOWED_VALUES 'student-success', 'advising', 'institutional-research'
  COMMENT = 'Cost-centre attribution for AI and warehouse spend';
ALTER AGENT STUDENT360_LAB.AGENTS.ADVISOR_ASSISTANT       SET TAG STUDENT360_LAB.AGENTS.COST_CENTER = 'advising';
ALTER AGENT STUDENT360_LAB.AGENTS.STUDENT_SUCCESS_ANALYST SET TAG STUDENT360_LAB.AGENTS.COST_CENTER = 'institutional-research';
ALTER WAREHOUSE STUDENT360_LAB_WH SET TAG STUDENT360_LAB.AGENTS.COST_CENTER = 'student-success';
SHOW TAGS IN SCHEMA STUDENT360_LAB.AGENTS;

In [ ]:
%%sql -r harden_warehouse
USE ROLE ACCOUNTADMIN;
ALTER WAREHOUSE STUDENT360_LAB_WH SET STATEMENT_TIMEOUT_IN_SECONDS = 300;
SHOW WAREHOUSES LIKE 'STUDENT360_LAB_WH';

In [ ]:
%%sql -r create_budget
USE ROLE STUDENT360_LAB_ADMIN;
CREATE OR REPLACE SNOWFLAKE.CORE.BUDGET STUDENT360_LAB.AGENTS.AI_BUDGET();
CALL STUDENT360_LAB.AGENTS.AI_BUDGET!SET_SPENDING_LIMIT(50);
CALL STUDENT360_LAB.AGENTS.AI_BUDGET!SET_NOTIFICATION_THRESHOLD(75);
CALL STUDENT360_LAB.AGENTS.AI_BUDGET!GET_SPENDING_LIMIT();

In [ ]:
%%sql -r create_quota
USE ROLE ACCOUNTADMIN;
CREATE OR REPLACE SNOWFLAKE.CORE.QUOTA STUDENT360_LAB.AGENTS.AI_USER_QUOTA();
CALL STUDENT360_LAB.AGENTS.AI_USER_QUOTA!ADD_SHARED_RESOURCE('CORTEX AGENT', (SELECT SYSTEM$REFERENCE('CORTEX AGENT', 'STUDENT360_LAB.AGENTS.ADVISOR_ASSISTANT')));
CALL STUDENT360_LAB.AGENTS.AI_USER_QUOTA!ADD_SHARED_RESOURCE('CORTEX AGENT', (SELECT SYSTEM$REFERENCE('CORTEX AGENT', 'STUDENT360_LAB.AGENTS.STUDENT_SUCCESS_ANALYST')));
CALL STUDENT360_LAB.AGENTS.AI_USER_QUOTA!SET_PER_USER_LIMIT(50);
CALL STUDENT360_LAB.AGENTS.AI_USER_QUOTA!GET_CONFIG();

### 🔹 Why monitoring mode for the quota

The quota in this workshop is **created in monitoring mode** - it records and reports usage, but it does not block anyone or send emails. That keeps the lab safe to run in a shared account. In production, set a per-user daily limit and enforcement; the call is the same with an extra enforcement parameter.

## 🧠 What You Just Built

| Control | Layer | Where |
|---|---|---|
| Two agents, each scoped by role | Access (layer 2) | Step 4-5 |
| Guardrail: contact details refused, adverse action refused | Instructions + `AI_QUESTION_CATEGORIZATION` | Step 5 |
| Budget per agent (tokens + seconds) | Orchestration | Step 2-3 |
| Attribution tags | Cost | Step 6 |
| Warehouse timeout | Cost | Step 6 |
| Spending budget with alert | Cost | Step 6 |
| Per-user quota | Cost | Step 6 |

## ✅ Notebook 04 complete

### What you just built
- Agents `STUDENT360_LAB.AGENTS.ADVISOR_ASSISTANT` (Engineering) and `STUDENT360_LAB.AGENTS.STUDENT_SUCCESS_ANALYST` (institution-wide)
- Layer-2 `USAGE ON AGENT` grants to each persona's role
- Cost attribution tags, a warehouse statement timeout, a spending budget and a per-user quota (monitoring mode)

---
### Key points
- An agent is a versioned spec: instructions, tools and budgets in one object.
- Same code, different persona: the data layer decides what each agent can answer.
- Observe, attribute, control - AI spend needs the same discipline as compute.

---
### Next
**Notebook 05** - evaluate the agents with a scored gate and publish them to CoWork.